### Convolution Neural Networks

This notebook is modified from: 
       https://www.tomasbeuzen.com/deep-learning-with-pytorch/chapters/chapter5_cnns-pt1.html
   
Further references: 

1. Conv2D: https://docs.pytorch.org/docs/2.12/generated/torch.nn.Conv2d.html
2. visualization:
       https://setosa.io/ev/image-kernels/


import the libraries

In [ ]:
import torch
#import torchvision
#from torchvision.transforms import v2
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
from torchinfo import summary
from sklearn.datasets import load_sample_image

For illustration, we load an image from sklearn package.

In [ ]:
flower = load_sample_image('flower.jpg')

plt.imshow(flower, cmap='gray')

# 1. change the shape such that the first dimension is channel (color)
# 2. convert to PyTorch tensor
img = torch.tensor(np.transpose(flower, (2,0,1)), dtype=torch.float)

print ('before:', flower.shape)
print ('after:', img.shape)

In [ ]:
cmap = [plt.cm.Reds, plt.cm.Greens, plt.cm.Blues]

fig, ax = plt.subplots(1,4, figsize=(12,3))
ax[0].imshow(flower)
ax[0].axis('off')
ax[0].set_title('original')
for i in range(3):
    ax[i+1].imshow(flower[:,:,i], cmap=cmap[i])
    ax[i+1].axis('off')
    ax[i+1].set_title(f'channel {i+1}')
plt.tight_layout()

fig.savefig('image_data.png')

a small function to visualize the image

In [ ]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    plt.imshow(np.transpose(img, (1,2,0)), cmap='gray')
    plt.show()

## Ingradient 1. Convolution Layer

#### Test 1: Define a convolution layer

Convolution really just means "to pass over the data" using filters, which are also called **kernels**. 

We can easily define a convolution network using PyTorch.

Pay attention to how the shape of the network parameters depends on the input parameters.


In [ ]:
in_channels = 3
out_channels = 6
kernel_size = 4

conv1 = nn.Conv2d(in_channels, out_channels, kernel_size)

# Print the network and its parameters

print (conv1)

print ("\n weight:", conv1.weight.shape, '\n bias:', conv1.bias.shape)

We can now apply the convolution to the image. 

How does the output's shape change when the network's parameters changes? 

In [ ]:
out = conv1(img).detach().numpy()

print (out.shape)

Let's have a look at the first channel of the output.

In [ ]:
imshow(out[:1])

#### Test 2: More extensive tests

Let's first write two functions to display the outputs of convolutions.

In [ ]:
# apply convolution with the kernel defined by the input matrix (filter).

def plot_conv(image, filter):
    """Plot convs with matplotlib."""
    d = filter.shape[-1]
    with torch.no_grad():    
        conv = torch.nn.Conv2d(1, 1, kernel_size=(d, d), padding=1)
        conv.weight[:] = filter
        
    fig, (ax1, ax2) = plt.subplots(figsize=(8, 4), ncols=2)
    ax1.imshow(image[0], cmap='gray')
    ax1.axis('off')
    ax1.set_title("Original")
    
    filtered = conv(image[:1]).detach().squeeze()
    ax2.imshow(filtered, cmap='gray')  
    ax2.set_title("Filtered")
    ax2.axis('off')
    plt.tight_layout();

# apply convolution with the kernel given by the input.

def plot_convs(image, conv_layer, axis=False):
    """Plot convs with matplotlib. Sorry for this lazy code :D"""
    filtered_image = conv_layer(image)
    n = filtered_image.shape[0]
    print ('Total filters:', n)
    print ('output shape:', filtered_image[0].shape)
    if n == 1:
        fig, (ax1, ax2) = plt.subplots(figsize=(8, 4), ncols=2)
        ax1.imshow(image[0], cmap='gray')
        ax1.set_title("Original")
        ax2.imshow(filtered_image.detach().squeeze(), cmap='gray')  
        ax2.set_title("Filter 1")
        ax1.grid(False)
        ax2.grid(False)
        if not axis:
            ax1.axis(False)
            ax2.axis(False)
        plt.tight_layout();
    elif n == 2:
        filtered_image_1 = filtered_image[0,:,:]
        filtered_image_2 = filtered_image[1,:,:]
        fig, (ax1, ax2, ax3) = plt.subplots(figsize=(10, 4), ncols=3)
        ax1.imshow(image[0], cmap='gray')
        ax1.set_title("Original")
        ax2.imshow(filtered_image_1.detach().squeeze(), cmap='gray')  
        ax2.set_title("Filter 1")
        ax3.imshow(filtered_image_2.detach().squeeze(), cmap='gray')  
        ax3.set_title("Filter 2")
        ax1.grid(False)
        ax2.grid(False)
        ax3.grid(False)
        plt.tight_layout();
    elif n >= 3:    
        filtered_image_1 = filtered_image[0,:,:]
        filtered_image_2 = filtered_image[1,:,:]
        filtered_image_3 = filtered_image[2,:,:]
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(figsize=(12, 4), ncols=4)
        ax1.imshow(image[0], cmap='gray')
        ax1.set_title("Original")
        ax2.imshow(filtered_image_1.detach().squeeze(), cmap='gray')  
        ax2.set_title("Filter 1")
        ax3.imshow(filtered_image_2.detach().squeeze(), cmap='gray')  
        ax3.set_title("Filter 2")
        ax4.imshow(filtered_image_3.detach().squeeze(), cmap='gray')  
        ax4.set_title("Filter 3")
        ax1.grid(False)
        ax2.grid(False)
        ax3.grid(False) 
        ax4.grid(False)
        if not axis:
            ax1.axis(False)
            ax2.axis(False)
            ax3.axis(False)
            ax4.axis(False)
        plt.tight_layout();        


Take a look at the outputs of different filters. 

Only the outputs of the first 3 filters are shown.

In [ ]:
print (img.shape)
plot_convs(img, conv1)

Let's try a customized kernel defined as a $3\times 3$ matrix.

In [ ]:

kernel = torch.tensor([[[[ 0.0625,  0.1250,  0.0625],
                         [ 0.1250,  0.2500,  0.1250],
                         [ 0.0625,  0.1250,  0.0625]]]])
plot_conv(img, kernel)

Try another matrix.

In [ ]:
kernel = torch.tensor([[[[ -2,  -1,  0],
                         [ -1,   1,  1],
                         [  0,   1,  2]]]])
plot_conv(img, kernel)

One more

In [ ]:
kernel = torch.tensor([[[[  -1,  -1,   -1],
                         [  -1,   8,   -1],
                         [  -1,  -1,   -1]]]])
plot_conv(img, kernel)

In [ ]:
# 1 kernel of (3,3) 
img_gray = img[:1]
conv_layer = torch.nn.Conv2d(1, 1, kernel_size=(3, 3))
plot_convs(img_gray, conv_layer)

How does the size of output image change?

In [ ]:
# 2 kernel of (5,5) 
img_gray = img[:1]
conv_layer = torch.nn.Conv2d(1, 2, kernel_size=(5, 5))
plot_convs(img_gray, conv_layer, axis=True)

By default, kernels are only applied where the filter fully fits on top of the input. But we can control this behaviour and the size of our output with:
- `padding`: "pads" the outside of the input 0's to allow the kernel to reach the boundary pixels
- `strides`: controls how far the kernel "steps" over pixels.

Below is an example with:
- `padding=1`: we have `1` layer of 0's around our border
- `strides=(2,2)`: our kernel moves 2 data points to the right for each row, then moves 2 data points down to the next row


Setting `padding = kernel_size // 2` will always result in an output the same shape as the input. Think about why this is...

In [ ]:
# 2 kernel of (5,5) with padding

conv_layer = torch.nn.Conv2d(1, 2, kernel_size=(5, 5), padding=2)

plot_convs(img_gray, conv_layer, axis=True)

**stride** also affects the size of the output.

In [ ]:
# 1 kernel of (5,5) with stride of 2
conv_layer = torch.nn.Conv2d(1, 1, kernel_size=(5, 5), stride=2)
plot_convs(img_gray, conv_layer, axis=True)

### Ingredient 2: Flatterning

With the convolutional layers, we're only passing images through the network.

But we're going to eventually want to do some regression or classification. That means that by the end of our network, we are going to need to `torch.nn.Flatten()` our images.

See: https://docs.pytorch.org/docs/2.12/generated/torch.nn.modules.flatten.Flatten.html

Let's make that simple CNN above in PyTorch:

why 546560 in the final layer?

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.main = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=1, out_channels=3, kernel_size=(3, 3), padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=3, out_channels=2, kernel_size=(3, 3), padding=1),
            torch.nn.ReLU(),
            torch.nn.Flatten(start_dim=0),
            torch.nn.Linear(546560, 1)
        )

    def forward(self, x):
        out = self.main(x)
        return out

In [ ]:
model = CNN()

img_gray_shape = img_gray.shape  # (1, 427, 640)

summary(model, img_gray_shape)

verify that our model maps an image to a scalar.

In [ ]:
out = model(img_gray)

print ('input shape:', img_gray.shape)
print ('output=', out)

### Ingredient 3: Pooling

Pooling can reduce the number of parameters. It's very simple, we just aggregate the data, using the maximum or average of a window of pixels. 

We use "pooling layers" to reduce the shape of our image as it's passing through the network. 

So when we eventually torch.nn.Flatten(), we'll have less features in that flattened layer.

We can implement pooling with torch.nn.MaxPool2d(). 

Let's try it out and reduce the number of parameters.

Reference:
    https://docs.pytorch.org/docs/2.12/generated/torch.nn.MaxPool2d.html

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.main = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=1, out_channels=3, kernel_size=(3, 3), padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d((2, 2)),
            torch.nn.Conv2d(in_channels=3, out_channels=2, kernel_size=(3, 3), padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d((2, 2)),
            torch.nn.Flatten(start_dim=0),
            # with pooling, the number of nodes reduces from 2048 to 128.
            torch.nn.Linear(33920, 1)
        )

    def forward(self, x):
        out = self.main(x)
        return out

check the number of parameters is less than before.

In [ ]:
model = CNN()

img_gray_shape = img_gray.shape  

summary(model, img_gray_shape)